In [1]:
%%capture
import sys

# Añade el directorio principal al path de búsqueda para importar módulos desde esa ubicación
sys.path.insert(0, "..")

# Desactivar los warnings para evitar mensajes innecesarios durante la ejecución
import warnings

import math
import numpy as np
import pandas as pd
from likelihood.models.ensemble import EnsembleClassifier
from likelihood import Pipeline

In [2]:
df = pd.read_csv("train.csv")
df["Heart Disease"] = df["Heart Disease"].replace({"Presence": 1, "Absence": 0})
df["Sex"] = df["Sex"].astype("category")
etl_pipe = Pipeline("ensemble_config.json")
x_train, y_train, importances = etl_pipe.fit(df.copy().drop(columns=["id"]))
X_train = np.asarray(x_train.to_numpy()).astype(np.float32)
y_train = y_train.reshape((y_train.size, 1))
_train = (np.eye(y_train.max() + 1)[y_train]).reshape((-1, 2))
y_train = np.asarray(_train).astype(np.float32)

df_test = pd.read_csv("test.csv")
df_test["Sex"] = df_test["Sex"].astype("category")
X_test = etl_pipe.transform(df_test.copy().drop(columns=["id"]))
X_test.insert(0, "id", df_test["id"])
X_test = np.asarray(X_test.drop(columns=["id"]).to_numpy()).astype(np.float32)

In [3]:
# Define parameter ranges for variation
param_ranges = {
    "units": (10, 20),
    "activation": ["selu", "relu"],
    "num_layers": (1, 5),
    "dropout": (0.0, 0.5),
}

# Create and train the ensemble
ensemble = EnsembleClassifier(
    n_models=2, param_ranges=param_ranges, seed_range=(0, 100), voting_method="soft", verbose=1
)

ensemble.fit(X_train, y_train, epochs=1, validation_split=0.2)
ensemble.save("./ensemble")
ensemble = EnsembleClassifier.load("./ensemble")

# Predictions
predictions = ensemble.predict(X_test)
probabilities = ensemble.predict_proba(X_test)

# Evaluate individual models
scores = ensemble.get_model_scores()
for score in scores:
    print(
        f"Model {score['model_id']}: F1={score['f1_score']:.3f}, Val Loss={score['val_loss']:.4f}"
    )

Training model 1/2...
Training model 2/2...
Ensemble trained with 2 models.
Model 1: F1=0.845, Val Loss=0.3362
Model 2: F1=0.842, Val Loss=0.3604


In [4]:
pred = ensemble.predict_proba(X_test)

df = pd.DataFrame(columns=["id", "Heart Disease"])
df["id"] = df_test["id"]
df["Heart Disease"] = pred[:, 1]
# truncate 1 decimal places
df["Heart Disease"] = df["Heart Disease"].apply(lambda x: float(math.floor(x * 10) / 10))

df.to_csv("sample_submission.csv", index=False)

In [5]:
ensemble.fit(X_train, y_train, epochs=1, validation_split=0.2)

# Evaluate individual models
scores = ensemble.get_model_scores()
for score in scores:
    print(
        f"Model {score['model_id']}: F1={score['f1_score']:.3f}, Val Loss={score['val_loss']:.4f}"
    )

Training model 1/2...
Training model 2/2...
Ensemble trained with 2 models.
Model 1: F1=0.855, Val Loss=0.3694
Model 2: F1=0.797, Val Loss=0.3046
